# 03 · CD: modelo físico

**Objetivo.** Mostrar y validar el modelo físico de la dispersión cromática (pasos B1 y B2 de la guía):

- la función de transferencia $H_{CD}$ con la **convención de signo positiva** (decisión 2), sin el factor ½ (decisión 3);
- las magnitudes derivadas: $\beta_2$, $\beta_3$, pendiente del retardo de grupo, largo del chirp $R$, tamaño de bloque $N$ y $L_D$;
- la **referencia ideal** (`cd_reference`, una FFT de toda la señal), que es la verdad física contra la que se va a medir el overlap-save (B3);
- su efecto sobre un pulso gaussiano y sobre una señal QAM16.

Todo sale de `emulador/cd/` (`physics.py`, `reference.py`), el mismo código que usan los tests (`tests/test_chromatic_dispersion.py`). Referencias: `referencia_guia_ISI_CD.md` B.1 a B.3; PDF *Fundamentos* §4.1 a §4.7.1 (Ec. 16 a 26, Figs. 4 a 6, Tablas 6 y 15).

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from emulador import plots
from emulador.cd import betas, cd_N, cd_R, cd_reference, cd_reference_pad, cd_summary, cd_transfer
from emulador.config import C, D_si, S_si, Scenario
from emulador.metrics import group_delay, nrmse, rms_width
from emulador.rx import recover
from emulador.stimuli import centered_time, comm_signal, gaussian_pulse

plots.use_style()

## 1. Teoría

**Transferencia.** Desarrollando la constante de propagación alrededor de la portadora (sin $\beta_0$ ni $\beta_1$):

$$
H_{CD}(\omega) = \exp\!\left[-j\left(\frac{\beta_2}{2}\omega^2 + \frac{\beta_3}{6}\omega^3\right)z\right],
\qquad
\beta_2 = -\frac{D\lambda^2}{2\pi c},
\qquad
\beta_3 = \left(\frac{\lambda^2}{2\pi c}\right)^2\left(S + \frac{2D}{\lambda}\right).
$$

Con $\beta_3 = 0$ y la convención positiva queda $H_{CD}(f) = \exp\!\left(+j\,\pi D\lambda^2 z f^2/c\right)$. Es un **pasa-todo**: $|H_{CD}| = 1$, la fase es una parábola en $f$ y el retardo de grupo es una recta,

$$
\tau(f) = -\frac{1}{2\pi}\frac{d\phi}{df} = -\frac{D\lambda^2 z}{c}\,f .
$$

Con $D > 0$ las frecuencias altas llegan antes. El compensador (tesis de CDC) tiene que usar $H_{CD}^*$. Además $H(z_1)H(z_2) = H(z_1+z_2)$ y $H^{-1} = H^*$.

**Pulso gaussiano.** Para $A(0,t) = \exp(-t^2/2T_0^2)$, el ancho crece como $T(z) = T_0\sqrt{1 + (z/L_D)^2}$, con $L_D = T_0^2/|\beta_2|$ (Ec. 22 y 23).

**Largo del chirp.** La respuesta al impulso es un chirp infinito; la parte que cae dentro de la banda de Nyquist ocupa

$$
R = 2\left\lfloor \frac{|D|\lambda^2 z}{2cT_s^2}\right\rfloor + 1
$$

muestras (Ec. 26). El overlap-save necesita bloques de $N \ge R$ (decisión 5); $R$ crece con $z$ y con $f_s^2$.

**Referencia ideal.** `cd_reference` rellena la señal con $R + 64$ ceros a cada lado y aplica $H_{CD}$ con una sola FFT. Tiene un piso numérico de NRMSE ~1e-6 (decisión 18): las colas del chirp discreto decaen como ~$1/n$ y no entran del todo en el relleno.

## 2. Parámetros

In [ ]:
esc = Scenario()        # 32 GBd, 2 sps -> fs = 64 GSa/s, 1550 nm
fs, lam = esc.fs, esc.wavelength
D = D_si(17.0)          # SSMF: 17 ps/(nm·km)
S = S_si(0.057)         # 0,057 ps/(nm²·km); se usa solo donde se indica
KM = 1e3
N_SYM = 4096
SEED = 1

esc.to_dict()

## 3. Magnitudes derivadas (Tabla 15 del PDF)

Para SSMF a 1550 nm y 64 GSa/s. $N$ no se acota a `N_max`; `N_fits_hw` indica si entra en el hardware provisorio (`HardwareParams`).

In [ ]:
print("  z [km]  beta2 [ps²/km]  beta3 [ps³/km]  pend. [ps/GHz]  Dz [ps/nm]      R      N  entra  L_D [km]")
summaries = {}
for z_km in (10, 50, 100, 500, 1000, 2000):
    s = cd_summary(esc, D, z_km * KM, S)
    summaries[z_km] = s
    print(f"{z_km:8d}  {s['beta2_ps2_km']:14.2f}  {s['beta3_ps3_km']:14.4f}  {s['gd_slope_ps_GHz']:14.2f}"
          f"  {s['Dz_ps_nm']:10.0f}  {s['R']:5d}  {s['N']:5d}  {str(s['N_fits_hw']):>5}  {s['L_D_km']:8.1f}")

## 4. Función de transferencia (Fig. 4 del PDF)

Magnitud, fase desenrollada y retardo de grupo medido con `metrics.group_delay` para 10, 50 y 100 km, con $\beta_3 = 0$. La línea punteada es la recta teórica $-D\lambda^2 z f/c$.

In [ ]:
L = 4096
Z_TF = (10, 50, 100)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
edge_delay = {}
for z_km in Z_TF:
    z = z_km * KM
    H = cd_transfer(L, fs, D, lam, z)
    f = np.fft.fftshift(np.fft.fftfreq(L, 1 / fs))
    Hs = np.fft.fftshift(H)
    f_gd, tau = group_delay(H, fs)
    axs[0].plot(f / 1e9, np.abs(Hs), label=f"{z_km} km")
    phi = np.unwrap(np.angle(Hs))
    axs[1].plot(f / 1e9, phi - phi[L // 2], label=f"{z_km} km")  # referida a f = 0
    line, = axs[2].plot(f_gd / 1e9, tau / 1e-12, label=f"{z_km} km")
    axs[2].plot(f_gd / 1e9, -D * lam**2 * z * f_gd / C / 1e-12, ":", color=line.get_color())
    edge_delay[z_km] = -D * lam**2 * z * (-fs / 2) / C
axs[0].set_ylim(0, 1.1)
axs[0].set_title("Magnitud $|H_{CD}(f)|$")
axs[1].set_title("Fase $\\phi(f) - \\phi(0)$ [rad]")
axs[2].set_title("Retardo de grupo $\\tau(f)$ [ps]")
for ax in axs:
    ax.set_xlabel("f [GHz]")
    ax.legend()
plots.save_fig(fig, "03_transferencia")

for z_km in Z_TF:
    print(f"{z_km:4d} km: retardo en el borde de la banda (f = -fs/2) = {edge_delay[z_km] / 1e-12:6.1f} ps")

**Qué muestra la figura.** $|H_{CD}| = 1$ para toda distancia: es un pasa-todo. La fase es una parábola de curvatura **positiva** (convención de la decisión 2) que crece con z: 4,4 / 22 / 44 rad en el borde de la banda a 10 / 50 / 100 km. El retardo de grupo es una recta de pendiente $-D\lambda^2 z/c$ (−13,6 ps/GHz a 100 km, ±436 ps en el borde de la banda), superpuesta a la recta teórica punteada: con $D > 0$ las frecuencias altas llegan antes.

## 5. Largo del chirp $R$ (Tabla 6 del PDF)

$R$ crece linealmente con $z$ y con $f_s^2$. A 40 GSa/s los valores coinciden con Jiang et al. (2026).

In [ ]:
Z_R = (10, 100, 500, 800, 1000, 2000)
R_table = {fs_: [cd_R(fs_, D, lam, z_km * KM) for z_km in Z_R] for fs_ in (40e9, 64e9)}
print("z [km]       " + "".join(f"{z:7d}" for z in Z_R))
for fs_, Rs_ in R_table.items():
    print(f"R a {fs_ / 1e9:.0f} GSa/s " + "".join(f"{r:7d}" for r in Rs_))
print("N a 64 GSa/s " + "".join(f"{cd_N(fs, D, lam, z * KM):7d}" for z in Z_R))

## 6. Pulso gaussiano (Fig. 5 del PDF)

$T_0 = 15{,}6$ ps ($L_D = 11{,}2$ km). Se muestrea a 256 GSa/s: a 64 GSa/s $T_0 \approx T_s$ y el aliasing agrega un error de ~0,2 % al ancho.

In [ ]:
T0, fs_g, n_g = 15.6e-12, 256e9, 4096
b2, _ = betas(D, lam)
L_D = T0**2 / abs(b2)
t = centered_time(n_g, fs_g)
a0 = gaussian_pulse(T0, fs_g, n_g)
w0 = rms_width(a0, t)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.3))
for z_km in (0, 25, 50, 100):
    az = cd_reference(a0, fs_g, D, lam, z_km * KM)
    axs[0].plot(t / 1e-12, np.abs(az) ** 2, label=f"{z_km} km")
axs[0].set_xlim(-200, 200)
axs[0].set_xlabel("t [ps]")
axs[0].set_title(f"$|A(z,t)|^2$, $T_0$ = {T0 / 1e-12:.1f} ps")
axs[0].legend()

z_sim = np.arange(0, 101, 5)
ratio_sim = np.array([rms_width(cd_reference(a0, fs_g, D, lam, z * KM), t) / w0 for z in z_sim])
z_th = np.linspace(0, 100, 201)
axs[1].plot(z_th, np.sqrt(1 + (z_th * KM / L_D) ** 2), label="teoría $\\sqrt{1+(z/L_D)^2}$")
axs[1].plot(z_sim, ratio_sim, "o", label="simulado (cd_reference)")
axs[1].set_xlabel("z [km]")
axs[1].set_title(f"$T(z)/T_0$ ($L_D$ = {L_D / KM:.1f} km)")
axs[1].legend()
plots.save_fig(fig, "03_pulso_gaussiano")

err_gauss = np.max(np.abs(ratio_sim / np.sqrt(1 + (z_sim * KM / L_D) ** 2) - 1))
print(f"L_D = {L_D / KM:.2f} km;  error relativo máximo del ancho = {err_gauss:.1e}")

**Qué muestra la figura.** A la izquierda, el pulso se ensancha y su pico baja (1 → 0,41 → 0,22 → 0,11 a 0 / 25 / 50 / 100 km) sin perder energía. A la derecha, el ancho simulado con `cd_reference` cae sobre la curva teórica $\sqrt{1+(z/L_D)^2}$ con un error de ~1e-15 ($L_D$ = 11,2 km; a 100 km el pulso es ~9 veces más ancho). Valida el modelo físico completo: signo, magnitud de β₂ y ausencia del factor ½.

## 7. Señal QAM16 (Fig. 6 del PDF)

QAM16 a 32 GBd, 2 sps, RRC con roll-off 0,1, después del receptor mínimo (`rx.recover`: filtro adaptado, fase de muestreo, retardo y ganancia compleja óptimos). El receptor corrige la rotación aparente que produce la CD, pero no la ISI. El último caso aplica 500 km y después la inversa ($-D$, $-S$), con β₃ incluido: verifica la reversibilidad del modelo.

In [ ]:
sig = comm_signal(esc, mod="qam16", n_symbols=N_SYM, seed=SEED)
x = sig.samples

cases = {"Sin CD": x}
for z_km in (1, 5):
    cases[f"CD {z_km} km"] = cd_reference(x, fs, D, lam, z_km * KM)
z500 = 500 * KM
y500 = cd_reference(x, fs, D, lam, z500, S=S, full=True)
back = cd_reference(y500, fs, -D, lam, z500, S=-S, full=True)
off = cd_reference_pad(fs, D, lam, z500) + cd_reference_pad(fs, -D, lam, z500)
cases["CD 500 km + $H^*$"] = back[off : off + x.size]

fig, axs = plt.subplots(1, 4, figsize=(17, 4.6))
evm = {}
for ax, (name, y) in zip(axs, cases.items()):
    r = recover(y, sig)
    evm[name] = r.evm
    plots.constellation(ax, r.symbols, ref=r.ref, title=f"{name} · EVM {100 * r.evm:.2f} %")
plots.save_fig(fig, "03_constelaciones_qam16")

for name, e in evm.items():
    print(f"{name:20s} EVM = {100 * e:6.2f} %")
rev_nrmse = nrmse(cases["CD 500 km + $H^*$"], x)
print(f"NRMSE de 500 km + H* contra la entrada: {rev_nrmse:.1e}")

**Qué muestra la figura.** Con apenas 1 km de CD la EVM sube de 0,12 % a 3,3 %, y con 5 km a 16,4 %: cada símbolo recibido mezcla muchos símbolos transmitidos y los puntos se vuelven nubes (a distancias grandes, una nube gaussiana). Aplicar 500 km y después la inversa (−D, −S) devuelve la constelación original (EVM 0,12 %, NRMSE 2,8e-6): el modelo es reversible hasta su piso numérico (decisión 18).

## 8. Validación

Los mismos criterios que `tests/test_chromatic_dispersion.py`.

In [ ]:
s100 = summaries[100]
assert abs(s100["beta2_ps2_km"] / -21.7 - 1) < 0.01
assert abs(s100["beta3_ps3_km"] / 0.13 - 1) < 0.03
assert abs(s100["gd_slope_ps_GHz"] / -13.6 - 1) < 0.01
assert (s100["R"], s100["N"]) == (55, 128)
assert abs(s100["L_D_km"] / 45 - 1) < 0.01
assert R_table[40e9] == [3, 21, 109, 175, 217, 435]
assert R_table[64e9] == [5, 55, 279, 447, 559, 1117]
for z_km in Z_TF:
    assert np.max(np.abs(np.abs(cd_transfer(L, fs, D, lam, z_km * KM)) - 1)) < 1e-12
assert err_gauss < 0.01
assert rev_nrmse < 1e-5
assert evm["Sin CD"] < 0.005 and evm["CD 500 km + $H^*$"] < 0.005
assert evm["CD 1 km"] < evm["CD 5 km"]
print("OK")

## 9. Conclusiones

- $H_{CD}$ es un pasa-todo ($|H| = 1$) con fase parabólica de curvatura **positiva** (decisión 2) y retardo de grupo lineal de pendiente $-D\lambda^2 z/c$: −13,6 ps/GHz a 100 km, 436 ps en el borde de la banda de 64 GSa/s. Los valores reproducen la Tabla 15 del PDF ($\beta_2$ = −21,7 ps²/km, $\beta_3$ = 0,13 ps³/km, $R$ = 55, $L_D$ ≈ 45 km para $T_0 = 1/R_s$).
- $R$ reproduce la Tabla 6 a 40 y 64 GSa/s. A 64 GSa/s, $N$ = 128 alcanza hasta 229 km y 2000 km necesita $N$ = 2048, el máximo provisorio del hardware.
- La referencia ideal reproduce el ensanchamiento del pulso gaussiano con error relativo ~1e-15 a 256 GSa/s ($L_D$ = 11,2 km; a 100 km el pulso se ensancha ~9 veces).
- Sobre QAM16, 1 km de CD ya lleva la EVM de 0,12 % a 3,3 % y 5 km a 16 %. Aplicar 500 km y después la inversa ($-D$, $-S$) recupera la constelación (EVM 0,12 %, NRMSE 2,8e-6): el modelo es reversible hasta el piso numérico de la decisión 18.

Figuras en `docs/golden_model/figuras/03_*.png`.